In [2]:
## Load the data
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [3]:
# Generating ground truth
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
#!wget {PREFIX}/01-agentic-rag/code/rag_helper.py
#!wget {PREFIX}/04-evaluation/code/evaluation_utils.py

In [4]:
## Module Instructions:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [5]:
## For each page, build a JSON user prompt from its filename and content, 
# then call llm_structured with the Questions model. 
# Turn each returned question into a record labeled with the page's filename. 
# The call also returns the token usage, the same as in the lessons.

## Q1: Generating questions for all 72 pages costs money and takes time, so let's start small and generate questions for just the first 3 pages:

- `01-agentic-rag/lessons/01-intro.md`
- `01-agentic-rag/lessons/02-environment.md`
- `01-agentic-rag/lessons/03-rag.md`
Each call returns the token usage, which most LLM APIs report on the response object (e.g. `response.usage.input_tokens` / `prompt_tokens`).

What's the average number of input tokens across these 3 calls?

In [6]:
document_subset = documents[0:3]

In [7]:
## Structure output with pydantic
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]


In [8]:
## Load OpenAI items:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
## shape the document as json
from evaluation_utils import llm_structured_retry
import json

ground_truth = []
usages = []

for doc in document_subset:  # first 3 pages
    user_prompt = json.dumps(doc)
    result, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )
    for q in result.questions:
        ground_truth.append({"question": q, "document": doc["filename"]})
    usages.append(usage)

In [10]:
usages

[ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=128, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1148),
 ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=114, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1400),
 ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=1280, cache_write_tokens=0), output_tokens=114, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1867)]

In [11]:
avg_input_tokens = sum(u.input_tokens for u in usages) / len(usages)
print(f"Average input tokens: {avg_input_tokens}")

Average input tokens: 1353.0


## Q1 answer: 1400

### Full Ground Truth

In [12]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
#!wget {PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv

# Load it with pandas into a list of records called ground_truth. 
# Each record has a question and the filename of the page that should answer it.


In [13]:
import pandas as pd
ground_truth = pd.read_csv('ground-truth.csv')

In [14]:
ground_truth.head(2)

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md


In [15]:
# Searching through chucks
# Create the chunks:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [19]:
# Now rebuild the search from homework 2 over these chunks. 
# Build a text index (Index) and a vector index (VectorSearch), both keyed on filename. 
# Wrap each one in a function, text_search and vector_search, that takes a query and the number of results to return (5 by default).

In [ ]:
# Wrap the search call in a function for text search
from minsearch import Index

# Text search index
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

In [1]:
from embedder import Embedder
embed = Embedder('/workspaces/llm-zoomcamp-rag/llm-zoomcamp-onnx/models/Xenova/all-MiniLM-L6-v2')

ModuleNotFoundError: No module named 'embedder'

In [ ]:
import numpy as np

chunk_vectors = [embed.encode(chunk['content']) for chunk in chunks]
X = np.array(chunk_vectors)

In [ ]:
from minsearch import VectorSearch

vs_index = VectorSearch(keyword_fields=["filename"])
vs_index.fit(X, chunks)

In [ ]:
#For hybrid search, reuse the rrf function from homework 2:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)